# Projeto Fictus | Analise de Vendas — Bloco 3: Escalabilidade Operacional


---

## Pergunta Central do Bloco
> **Crescer fortalece ou fragiliza o negócio?**

---

## Contexto do Bloco

Após mapear a origem e a concentração da receita no Bloco 2, este módulo investiga a capacidade de suporte da operação. Independentemente do volume de vendas, todo sistema possui um ponto de saturação. O foco aqui é identificar se o crescimento está gerando ganhos de eficiência ou se está degradando os indicadores de entrega e custo unitário.

Um negócio verdadeiramente escalável absorve novos volumes sem perder qualidade. Identificar o "ponto de inflexão" onde a operação começa a falhar é vital para que o comprador saiba exatamente quanto investimento em infraestrutura será necessário no dia seguinte à compra.

**Este bloco investiga:**
1. O custo médio por pedido aumenta, reduz ou se mantém estável com o crescimento do volume?
2. O lead time se deteriora em períodos de maior demanda?
3. Há correlação clara entre aumento de volume e falhas operacionais?
4. Quais gargalos emergem quando a operação é pressionada?
5. Em que ponto o crescimento passa a gerar ineficiência operacional?
6. Os gargalos operacionais são estruturais ou sazonais?
7. O atraso está na preparação do pedido ou na entrega final?
8. A degradação operacional em períodos de pico afeta todas as regiões igualmente — ou há regiões que colapsam antes das outras?

---

## Nota Metodológica — Deslocamento Temporal
Dados originais Olist **2016–2018** deslocados **+7 anos** → período **2023–2025**.  
**Análise restrita a partir de janeiro/2024** — dados de 2023 desconsiderados.

---

## Configuração do Ambiente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
from pathlib import Path

# ─── Caminhos relativos — funcionam em qualquer máquina ─────────────────────
# O notebook está em notebooks/ → a raiz é um nível acima
NOTEBOOK_DIR = Path().resolve()
# Detecta a raiz do projeto subindo a hierarquia de pastas
# Funciona em qualquer estrutura: raiz/, notebooks/, notebooks/vendas/
def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
BASE_DIR = _find_base(NOTEBOOK_DIR)
DIR_PRE      = BASE_DIR / "data" / "pre-tratados"
DIR_EXT      = BASE_DIR / "data" / "externos"
DIR_EXPORTS  = BASE_DIR / "exports"
DIR_EXPORTS.mkdir(parents=True, exist_ok=True)


warnings.filterwarnings("ignore")

COR_RECEITA  = "#1B4F72"
COR_MARGEM   = "#27AE60"
COR_ALERTA   = "#C0392B"
COR_NEUTRO   = "#7F8C8D"
COR_DESTAQUE = "#E67E22"
COR_ROXO     = "#8E44AD"
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})
def fmt_pct(x, pos=None):  return f"{x:.1f}%"
def fmt_dias(x, pos=None): return f"{x:.0f}d"
def salvar(fig, nome):
    caminho = DIR_EXPORTS / f"{nome}.png"
    fig.savefig(caminho)
    print(f"  → Salvo: {caminho.name}")
print("Ambiente configurado.")

## Carregamento dos Dados

In [ ]:
# ─── Separador padrão: vírgula, decimal ponto ────────────────────────────────
#   Todos os arquivos gerados pelo ETL usam sep=',' e decimal='.'
def ler_csv(caminho, sep=",", **kwargs):
    df = pd.read_csv(caminho, sep=sep, low_memory=False, **kwargs)
    df.columns = df.columns.str.strip()
    return df

fato  = ler_csv(DIR_PRE / "fato_vendas.csv")
dim_p = ler_csv(DIR_PRE / "dim_produto.csv")
dim_c = ler_csv(DIR_PRE / "dim_cliente.csv")
dim_v = ler_csv(DIR_PRE / "dim_vendedor.csv")
dim_t = ler_csv(DIR_PRE / "dim_tempo.csv")

# ─── Conversão de datas ───────────────────────────────────────────────────────
for col in ["data_compra", "data_entrega_cliente", "data_previsao_entrega",
            "data_envio_transportadora", "data_aprovacao"]:
    if col in fato.columns:
        fato[col] = pd.to_datetime(fato[col], errors="coerce")
dim_t["data"] = pd.to_datetime(dim_t["data"], errors="coerce")

# ─── Colunas numéricas: garantia adicional ────────────────────────────────────
for col in ["preco", "valor_frete", "valor_total_item", "valor_pagamento_total",
            "lead_time_dias", "atraso_dias", "nota_review", "numero_parcelas"]:
    if col in fato.columns:
        fato[col] = pd.to_numeric(fato[col], errors="coerce")

# ─── Enriquecimento e filtro ─────────────────────────────────────────────────
fato = fato.merge(dim_p[["id_produto", "nome_categoria_produto"]], on="id_produto",  how="left")
fato = fato.merge(dim_c[["id_cliente", "estado_cliente"]],         on="id_cliente",  how="left")
fato = fato.merge(dim_v[["id_vendedor", "estado_vendedor"]],        on="id_vendedor", how="left")
fato = fato.merge(
    dim_t[["id_data", "ano", "mes", "trimestre", "nome_mes", "ano_mes"]],
    on="id_data", how="left"
)
fato["periodo"] = fato["ano"].astype(str) + "-Q" + fato["trimestre"].astype(str)

DATA_INICIO = "2024-01-01"
fato = fato[fato["data_compra"] >= DATA_INICIO].copy()

fe  = fato[fato["status_pedido"] == "entregue"].copy()
fne = fato[fato["status_pedido"] != "entregue"].copy()
periodos_ord = sorted(fato["periodo"].dropna().unique())

# ─── Decomposição do lead time (preparação + transporte) ─────────────────────
if "data_aprovacao" in fe.columns and "data_envio_transportadora" in fe.columns:
    fe["tempo_preparacao"] = (fe["data_envio_transportadora"] - fe["data_aprovacao"]).dt.days
    fe["tempo_transporte"] = (fe["data_entrega_cliente"] - fe["data_envio_transportadora"]).dt.days
    tem_decomposicao = (
        fe["tempo_preparacao"].notna().sum() > 100 and
        fe["tempo_transporte"].notna().sum() > 100
    )
else:
    tem_decomposicao = False

print(f"[FILTRO] Dados a partir de: {DATA_INICIO}")
print(f"fato (filtrado)   : {len(fato):>8} linhas | {fato['data_compra'].min().date()} → {fato['data_compra'].max().date()}")
print(f"fato_entregues    : {len(fe):>8} linhas ({len(fe)/len(fato)*100:.1f}% do total)")
print(f"Lead time mediano : {fe['lead_time_dias'].median():.0f} dias")
print(f"Entregues no prazo: {fe['entregue_no_prazo'].mean()*100:.1f}%")
print(f"Decomposição LT   : {'disponível' if tem_decomposicao else 'colunas insuficientes'}")


---

## Análise 1 — O custo médio por pedido aumenta, reduz ou se mantém estável com o crescimento do volume?

> *"Em operações escaláveis, o ganho de escala deve, teoricamente, reduzir o custo unitário. Se o custo médio de frete por pedido acompanha ou supera o crescimento do volume, a operação não está capturando eficiência, mas sim repassando ineficiências logísticas ao cliente ou à margem. Esta análise quantifica a relação entre volume e custo para identificar se a expansão do negócio vem acompanhada de ganhos de produtividade ou de deterioração operacional."*

**Framework:** PDCA + Melhoria Contínua   
**Entrega:** Evolução do frete médio por pedido, % frete sobre receita e correlação volume × custo unitário

**Como este script responde à pergunta:**
> Em operações eficientes, o custo unitário deveria cair — ou ao menos se manter estável — conforme o volume cresce. O script calcula frete médio por pedido, percentual do frete sobre a receita e a correlação estatística entre volume mensal e frete unitário. Com esses três números, monta três painéis:
>
> 1. **Frete médio por pedido ao longo do tempo:** Plota a evolução mensal do frete unitário com uma linha de tendência e área sombreada em relação à média. A tendência calculada por regressão linear aparece anotada no gráfico — se subir, o custo unitário está crescendo com o negócio; se cair ou estabilizar, a operação está ganhando eficiência.
> 2. **% frete sobre receita por mês:** Barras em vermelho destacam os meses acima do terceiro quartil — os momentos em que o frete pesou mais sobre a receita. A linha tracejada marca a mediana do período como referência de normalidade.
> 3. **Scatter volume × frete médio:** Cada ponto é um mês. A linha de regressão e o coeficiente de correlação respondem diretamente à pergunta: quanto mais pedidos, o frete unitário sobe, cai ou fica igual? Correlação positiva significa que escalar o volume encarece o custo unitário — o oposto do esperado em uma operação eficiente.

**Análise do Resultado:**
Em uma operação escalável, esperamos ver o efeito de "ganho de escala": quanto mais vendemos, menor deve ser o custo unitário de frete, pois otimizamos rotas e aumentamos o poder de negociação. Se o gráfico mostrar que o custo de frete acompanha ou supera a subida do volume (correlação positiva), estamos diante de uma ineficiência logística. Isso indica que a empresa não está capturando escala, mas apenas "empilhando" pedidos em uma estrutura que já está no limite, o que é um sinal de alerta crítico para um investidor de M&A.

In [ ]:
frete_mensal = (
    fe.groupby("ano_mes")
    .agg(
        n_pedidos     = ("id_pedido",   "nunique"),
        frete_total   = ("valor_frete", "sum"),
        frete_medio   = ("valor_frete", "mean"),
        receita_total = ("preco",       "sum"),
    )
    .reset_index().sort_values("ano_mes")
)
frete_mensal["pct_frete_receita"] = frete_mensal["frete_total"] / frete_mensal["receita_total"] * 100
r_vol_frete, p_vol_frete = stats.pearsonr(frete_mensal["n_pedidos"], frete_mensal["frete_medio"])

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Análise 1 — Custo Unitário de Frete × Volume", fontsize=13, fontweight="bold")

x     = range(len(frete_mensal))
xtick = list(range(0, len(frete_mensal), 3))
xlabs = [frete_mensal["ano_mes"].iloc[i] for i in xtick]

axes[0].plot(x, frete_mensal["frete_medio"], color=COR_DESTAQUE, linewidth=2, marker="o", markersize=3)
axes[0].fill_between(x, frete_mensal["frete_medio"], frete_mensal["frete_medio"].mean(), alpha=0.15, color=COR_DESTAQUE)
axes[0].axhline(frete_mensal["frete_medio"].mean(), color=COR_NEUTRO, linestyle="--", linewidth=1,
                label=f"Média: R$ {frete_mensal['frete_medio'].mean():.2f}")
z = np.polyfit(list(x), frete_mensal["frete_medio"].fillna(method="ffill"), 1)
axes[0].plot(x, np.poly1d(z)(list(x)), color="black", linewidth=1, linestyle=":", alpha=0.6)
tend = "↑ crescendo" if z[0] > 0.01 else "↓ caindo" if z[0] < -0.01 else "→ estável"
axes[0].text(0.03, 0.93, f"Tendência: {tend}", transform=axes[0].transAxes,
             fontsize=8, color=COR_ALERTA if z[0] > 0.01 else COR_MARGEM)
axes[0].set_title("Frete Médio por Pedido (R$)", fontsize=11)
axes[0].set_ylabel("R$")
axes[0].set_xticks(xtick); axes[0].set_xticklabels(xlabs, rotation=45, ha="right", fontsize=8)
axes[0].legend(frameon=False, fontsize=8)

cores_pf = [COR_ALERTA if v > frete_mensal["pct_frete_receita"].quantile(0.75) else COR_NEUTRO
            for v in frete_mensal["pct_frete_receita"]]
axes[1].bar(x, frete_mensal["pct_frete_receita"], color=cores_pf, alpha=0.85)
axes[1].axhline(frete_mensal["pct_frete_receita"].median(), color=COR_DESTAQUE, linestyle="--", linewidth=1.5,
                label=f"Mediana: {frete_mensal['pct_frete_receita'].median():.1f}%")
axes[1].set_title("% Frete sobre Receita Mensal", fontsize=11)
axes[1].set_ylabel("%")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_xticks(xtick); axes[1].set_xticklabels(xlabs, rotation=45, ha="right", fontsize=8)
axes[1].legend(frameon=False, fontsize=8)

axes[2].scatter(frete_mensal["n_pedidos"], frete_mensal["frete_medio"],
                color=COR_RECEITA, alpha=0.7, edgecolors="white", linewidth=0.5, s=50)
m, b, _, _, _ = stats.linregress(frete_mensal["n_pedidos"], frete_mensal["frete_medio"])
xfit = np.linspace(frete_mensal["n_pedidos"].min(), frete_mensal["n_pedidos"].max(), 100)
axes[2].plot(xfit, m*xfit+b, color=COR_ALERTA, linewidth=1.5, linestyle="--")
sinal_frete = "mais volume = frete mais caro" if r_vol_frete > 0.3 else \
              "mais volume = frete mais barato" if r_vol_frete < -0.3 else "sem relação clara"
axes[2].set_title(f"Volume × Frete Médio\ncorr={r_vol_frete:.2f} — {sinal_frete}", fontsize=11)
axes[2].set_xlabel("Nº de Pedidos no Mês")
axes[2].set_ylabel("Frete Médio (R$)")

plt.tight_layout()
salvar(fig, "01_frete_volume")
plt.show()

print(f"\nFrete médio: R$ {fe['valor_frete'].mean():.2f} | {fe['valor_frete'].sum()/fe['preco'].sum()*100:.1f}% da receita")
print(f"Correlação volume × frete: {r_vol_frete:.2f} — {sinal_frete}")

---

## Análise 2 — O lead time se deteriora em períodos de maior demanda?

> *"A maturidade operacional de um ativo é medida pela estabilidade dos prazos de entrega sob diferentes níveis de estresse. A deterioração do lead time em períodos de alta demanda indica que o sistema opera próximo ao limite de sua capacidade instalada, sem margens de manobra. Para um investidor, este cenário sinaliza que qualquer aceleração de vendas pós-aquisição exigirá investimentos imediatos em infraestrutura para evitar a quebra do SLA."*

**Framework:** Lean Management — estabilidade de processo sob variação de demanda  
**Entrega:** Lead time médio, mediano e P90 ao longo do tempo, % no prazo e distribuição por quintil de volume

**Como este script responde à pergunta:**
> Lean Management define que um processo maduro mantém throughput constante sob variação de demanda. O script calcula lead time médio, mediano e P90 (o tempo que 90% dos pedidos levam) mês a mês, além do percentual entregue no prazo. Em seguida, segmenta os pedidos por quintil de volume mensal para ver se a distribuição do lead time se alarga quando o volume aumenta.
>
> 1. **Média, mediana e P90 ao longo do tempo:** As três linhas juntas revelam o comportamento da distribuição de lead time — se o P90 sobe muito mais que a média, os pedidos mais lentos estão ficando cada vez mais lentos, mesmo que a média pareça controlada. Isso é o sinal de uma fila se formando nos momentos de pressão.
> 2. **% entregue no prazo por mês:** Barras coloridas por nível de SLA — verde acima de 90%, laranja entre 80% e 90%, vermelho abaixo. As linhas de referência deixam visível quais meses ficaram fora do padrão mínimo aceitável.
> 3. **Scatter volume × lead time com cor de SLA:** Cada ponto é um mês, colorido pelo percentual de entregas no prazo. A linha de regressão e a correlação dizem se mais volume implica mais dias de entrega.
> 4. **Violin plot por quintil de volume:** Divide os meses em cinco grupos do menor para o maior volume e mostra a distribuição completa do lead time em cada grupo. Se o violino do Q5 (maior volume) for mais largo e deslocado para cima, a operação degrada sob pressão — evidência direta de fragilidade de escala.

**Análise do Resultado:** 
Este indicador mede a "folga" do sistema. Se o Lead Time (tempo de entrega) aumenta nos meses de pico, a operação trabalha sem margem de manobra. O ponto crucial aqui é o P90 (os 10% de pedidos mais lentos): se ele se descola muito da média durante picos de volume, o sistema está "engasgando" e criando filas. Para o comprador, isso significa que qualquer plano de expansão exigirá investimento imediato em novos centros de distribuição ou transportadoras.

In [ ]:
lt_mensal = (
    fe.groupby("ano_mes")
    .agg(
        n_pedidos    = ("id_pedido",         "nunique"),
        lead_medio   = ("lead_time_dias",    "mean"),
        lead_mediano = ("lead_time_dias",    "median"),
        lead_p90     = ("lead_time_dias",    lambda x: x.quantile(0.9)),
        pct_no_prazo = ("entregue_no_prazo", "mean"),
    )
    .reset_index().sort_values("ano_mes")
)
lt_mensal["pct_no_prazo"] = pd.to_numeric(lt_mensal["pct_no_prazo"], errors="coerce") * 100
mask_valid = lt_mensal["lead_medio"].notna() & lt_mensal["n_pedidos"].notna()
r_lt, p_lt = stats.pearsonr(
    lt_mensal.loc[mask_valid, "n_pedidos"],
    lt_mensal.loc[mask_valid, "lead_medio"]
)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("Análise 2 — Lead Time × Volume de Pedidos", fontsize=13, fontweight="bold")

x     = range(len(lt_mensal))
xtick = list(range(0, len(lt_mensal), 3))
xlabs = [lt_mensal["ano_mes"].iloc[i] for i in xtick]

axes[0,0].plot(x, lt_mensal["lead_p90"],    color=COR_ALERTA,   linewidth=1.5, linestyle="--", label="P90",    alpha=0.8)
axes[0,0].plot(x, lt_mensal["lead_medio"],  color=COR_DESTAQUE, linewidth=2,   label="Média",   marker="o", markersize=3)
axes[0,0].plot(x, lt_mensal["lead_mediano"],color=COR_MARGEM,   linewidth=1.5, linestyle="-.", label="Mediana", alpha=0.8)
axes[0,0].set_title("Lead Time: Média, Mediana e P90 (dias)", fontsize=11)
axes[0,0].set_ylabel("Dias")
axes[0,0].set_xticks(xtick); axes[0,0].set_xticklabels(xlabs, rotation=45, ha="right", fontsize=8)
axes[0,0].legend(frameon=False, fontsize=8)

cores_prazo = [COR_MARGEM if v >= 90 else COR_DESTAQUE if v >= 80 else COR_ALERTA for v in lt_mensal["pct_no_prazo"]]
axes[0,1].bar(x, lt_mensal["pct_no_prazo"], color=cores_prazo, alpha=0.85)
axes[0,1].axhline(90, color=COR_MARGEM,   linestyle="--", linewidth=1, alpha=0.7, label="SLA 90%")
axes[0,1].axhline(80, color=COR_DESTAQUE, linestyle=":",  linewidth=1, alpha=0.7, label="SLA mínimo 80%")
axes[0,1].set_title("% Pedidos Entregues no Prazo por Mês", fontsize=11)
axes[0,1].set_ylabel("%")
axes[0,1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0,1].set_xticks(xtick); axes[0,1].set_xticklabels(xlabs, rotation=45, ha="right", fontsize=8)
axes[0,1].legend(frameon=False, fontsize=8)

sc = axes[1,0].scatter(
    lt_mensal["n_pedidos"], lt_mensal["lead_medio"],
    c=lt_mensal["pct_no_prazo"], cmap="RdYlGn", vmin=70, vmax=100,
    s=60, alpha=0.8, edgecolors="white", linewidth=0.5
)
plt.colorbar(sc, ax=axes[1,0], label="% Entregue no prazo")
if mask_valid.sum() > 3:
    m2, b2, _, _, _ = stats.linregress(
        lt_mensal.loc[mask_valid, "n_pedidos"],
        lt_mensal.loc[mask_valid, "lead_medio"]
    )
    xfit_lt = np.linspace(lt_mensal["n_pedidos"].min(), lt_mensal["n_pedidos"].max(), 100)
    axes[1,0].plot(xfit_lt, m2*xfit_lt+b2, color="black", linewidth=1.5, linestyle="--", alpha=0.6)
axes[1,0].set_xlabel("Nº de Pedidos no Mês")
axes[1,0].set_ylabel("Lead Time Médio (dias)")
axes[1,0].set_title(f"Volume × Lead Time (corr={r_lt:.2f}, p={p_lt:.3f})", fontsize=11)

fe_lt = fe[["lead_time_dias", "ano_mes"]].dropna().copy()
vol_por_mes = fe.groupby("ano_mes")["id_pedido"].nunique()
fe_lt["vol_mes"] = fe_lt["ano_mes"].map(vol_por_mes)
fe_lt["quintil_vol"] = pd.qcut(fe_lt["vol_mes"], q=5,
                                labels=["Q1\n(menor)", "Q2", "Q3", "Q4", "Q5\n(maior)"])
sns.violinplot(
    data=fe_lt[fe_lt["lead_time_dias"] <= fe_lt["lead_time_dias"].quantile(0.99)],
    x="quintil_vol", y="lead_time_dias",
    palette=[COR_MARGEM, "#82E0AA", COR_DESTAQUE, "#F1948A", COR_ALERTA],
    ax=axes[1,1], inner="quartile", linewidth=0.8
)
axes[1,1].set_title("Distribuição do Lead Time por Quintil de Volume", fontsize=11)
axes[1,1].set_xlabel("Quintil de volume mensal")
axes[1,1].set_ylabel("Lead Time (dias)")

plt.tight_layout()
salvar(fig, "02_lead_time_volume")
plt.show()

interp_lt = (
    "RISCO: mais volume = entrega mais lenta"
    if r_lt > 0.3 else
    "POSITIVO: volume não degrada lead time"
    if r_lt < 0.1 else "ATENÇÃO: correlação moderada"
)
print(f"\nLead time médio: {fe['lead_time_dias'].mean():.1f}d | P90: {fe['lead_time_dias'].quantile(0.9):.1f}d")
print(f"% no prazo: {fe['entregue_no_prazo'].mean()*100:.1f}%")
print(f"Corr. volume × lead: {r_lt:.2f} — {interp_lt}")

---

## Análise 3 — Há correlação clara entre aumento de volume e falhas operacionais?

> *"Através da aplicação do pensamento sistêmico, investiga-se a relação de causa e efeito entre a escala e a taxa de erro. O aumento de falhas operacionais proporcional ao volume sugere que o sistema não foi projetado para a escala atual, operando em regime de sobrecarga. Identificar se existe uma relação estrutural entre volume e erro é crítico para avaliar a robustez dos processos internos."*

**Framework:** Análise de causa e efeito — Engenharia de Produção  
**Entrega:** Taxas de cancelamento e atraso ao longo do tempo + correlações com volume mensal

**Como este script responde à pergunta:**
> Falhas operacionais sob pressão de volume são o sinal mais direto de um sistema operando além da capacidade. O script calcula mês a mês a taxa de cancelamento (pedidos cancelados sobre total de pedidos) e a taxa de atraso (pedidos entregues após a data prevista sobre total de entregues), e mede a correlação de cada uma com o volume mensal.
>
> 1. **Taxa de cancelamento mensal:** Barras com linha de média. Picos acima da média em meses de alto volume são o sinal de que a operação não absorveu a demanda sem quebrar compromissos.
> 2. **Taxa de atraso mensal:** Mesma lógica, mas para entregas que saíram do prazo prometido ao cliente. Atraso é mais silencioso que cancelamento — o pedido chega, mas chega tarde, corroendo satisfação sem aparecer nas métricas de volume.
> 3. **Scatter volume × cancelamento:** A regressão e a correlação quantificam se o cancelamento é estrutural (independente do volume) ou induzido por demanda (piora quando o volume sobe).
> 4. **Scatter volume × atraso:** Idem para atraso. Correlação positiva e significativa (p < 0,05) confirma que o sistema perde confiabilidade quando pressionado — risco direto para qualquer plano de crescimento pós-aquisição.

**Análise do Resultado:**
 Analisamos se o crescimento "quebra" a operação. Um aumento na taxa de cancelamentos e atrasos proporcional ao volume de vendas prova que o sistema é frágil. Se a correlação for alta e positiva, o negócio sofre de um mal chamado anti-escala: quanto mais ele cresce, pior ele atende. Identificar se esse erro é estatisticamente significativo ajuda a separar erros humanos eventuais de uma falha estrutural de capacidade.

In [ ]:
falhas_mensal = (
    fato.groupby("ano_mes")
    .agg(
        n_total      = ("id_pedido", "nunique"),
        n_cancelados = ("status_pedido", lambda x: (x == "cancelado").sum()),
        n_entregues  = ("status_pedido", lambda x: (x == "entregue").sum()),
    )
    .reset_index().sort_values("ano_mes")
)
atrasos_mensal = (
    fe[fe["atraso_dias"] > 0].groupby("ano_mes")
    .agg(n_atrasados=("id_pedido", "nunique")).reset_index()
)
falhas_mensal = falhas_mensal.merge(atrasos_mensal, on="ano_mes", how="left")
falhas_mensal["n_atrasados"]   = falhas_mensal["n_atrasados"].fillna(0)
falhas_mensal["pct_cancelado"] = falhas_mensal["n_cancelados"] / falhas_mensal["n_total"] * 100
falhas_mensal["pct_atrasado"]  = falhas_mensal["n_atrasados"] / falhas_mensal["n_entregues"].replace(0, np.nan) * 100

r_vol_canc, p_vol_canc = stats.pearsonr(falhas_mensal["n_total"], falhas_mensal["pct_cancelado"])
mask_atr = falhas_mensal["pct_atrasado"].notna()
r_vol_atr, p_vol_atr = stats.pearsonr(
    falhas_mensal.loc[mask_atr, "n_total"],
    falhas_mensal.loc[mask_atr, "pct_atrasado"]
)

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle("Análise 3 — Volume × Falhas Operacionais", fontsize=13, fontweight="bold")

x     = range(len(falhas_mensal))
xtick = list(range(0, len(falhas_mensal), 3))
xlabs = [falhas_mensal["ano_mes"].iloc[i] for i in xtick]

axes[0,0].bar(x, falhas_mensal["pct_cancelado"], color=COR_ALERTA, alpha=0.8)
axes[0,0].axhline(falhas_mensal["pct_cancelado"].mean(), color=COR_NEUTRO, linestyle="--", linewidth=1.5,
                  label=f"Média: {falhas_mensal['pct_cancelado'].mean():.1f}%")
axes[0,0].set_title("Taxa de Cancelamento Mensal", fontsize=11)
axes[0,0].set_ylabel("%")
axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0,0].set_xticks(xtick); axes[0,0].set_xticklabels(xlabs, rotation=45, ha="right", fontsize=8)
axes[0,0].legend(frameon=False, fontsize=8)

axes[0,1].bar(x, falhas_mensal["pct_atrasado"].fillna(0), color=COR_DESTAQUE, alpha=0.8)
axes[0,1].axhline(falhas_mensal["pct_atrasado"].mean(), color=COR_NEUTRO, linestyle="--", linewidth=1.5,
                  label=f"Média: {falhas_mensal['pct_atrasado'].mean():.1f}%")
axes[0,1].set_title("Taxa de Atraso Mensal", fontsize=11)
axes[0,1].set_ylabel("%")
axes[0,1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0,1].set_xticks(xtick); axes[0,1].set_xticklabels(xlabs, rotation=45, ha="right", fontsize=8)
axes[0,1].legend(frameon=False, fontsize=8)

axes[1,0].scatter(falhas_mensal["n_total"], falhas_mensal["pct_cancelado"],
                  color=COR_ALERTA, alpha=0.7, edgecolors="white", linewidth=0.5, s=50)
m3, b3, _, _, _ = stats.linregress(falhas_mensal["n_total"], falhas_mensal["pct_cancelado"])
xfit_f = np.linspace(falhas_mensal["n_total"].min(), falhas_mensal["n_total"].max(), 100)
axes[1,0].plot(xfit_f, m3*xfit_f+b3, color="black", linewidth=1.5, linestyle="--", alpha=0.5)
axes[1,0].set_xlabel("Volume mensal")
axes[1,0].set_ylabel("% Cancelado")
axes[1,0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1,0].set_title(f"Volume × Cancelamento (corr={r_vol_canc:.2f})", fontsize=11)

fm_atr = falhas_mensal[mask_atr]
axes[1,1].scatter(fm_atr["n_total"], fm_atr["pct_atrasado"],
                  color=COR_DESTAQUE, alpha=0.7, edgecolors="white", linewidth=0.5, s=50)
if len(fm_atr) > 3:
    m4, b4, _, _, _ = stats.linregress(fm_atr["n_total"], fm_atr["pct_atrasado"])
    xfit4 = np.linspace(fm_atr["n_total"].min(), fm_atr["n_total"].max(), 100)
    axes[1,1].plot(xfit4, m4*xfit4+b4, color="black", linewidth=1.5, linestyle="--", alpha=0.5)
axes[1,1].set_xlabel("Volume mensal")
axes[1,1].set_ylabel("% Atrasado")
axes[1,1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1,1].set_title(f"Volume × Atraso (corr={r_vol_atr:.2f})", fontsize=11)

plt.tight_layout()
salvar(fig, "03_falhas_operacionais")
plt.show()

n_canc = fato[fato["status_pedido"] == "cancelado"]["id_pedido"].nunique()
n_atr  = fe[fe["atraso_dias"] > 0]["id_pedido"].nunique()

def interp_corr(r, p, m):
    sig = "(sig.)" if p < 0.05 else "(não sig.)"
    if r > 0.3 and p < 0.05:   return f"RISCO — crescimento piora {m} {sig}"
    elif r < -0.1 and p < 0.05: return f"POSITIVO — crescimento melhora {m} {sig}"
    return f"NEUTRO {sig}"

print(f"\nCancelados: {n_canc} ({n_canc/fato['id_pedido'].nunique()*100:.1f}%) | Atrasados: {n_atr} ({n_atr/fe['id_pedido'].nunique()*100:.1f}%)")
print(f"Corr. vol × cancelamento: {r_vol_canc:.2f} — {interp_corr(r_vol_canc, p_vol_canc, 'cancelamento')}")
print(f"Corr. vol × atraso      : {r_vol_atr:.2f} — {interp_corr(r_vol_atr, p_vol_atr, 'atraso')}")

---

## Análise 4 — Quais gargalos emergem quando a operação é pressionada?

> *"Fundamentada na Teoria das Restrições, esta análise identifica os pontos exatos onde o fluxo operacional é interrompido. Todo sistema possui um limitador; o objetivo é determinar se esse gargalo está mapeado e sob controle ou se ele se torna crítico conforme a demanda aumenta. Localizar a manifestação geográfica e temporal dessas restrições permite prever o limite real de expansão do negócio."*

**Framework:** Teoria das Restrições — identificação do gargalo do sistema  
**Entrega:** SLA por estado e por período, heatmap de entregas no prazo por estado × trimestre

**Como este script responde à pergunta:**
> A Teoria das Restrições diz que todo sistema tem um gargalo. Este script localiza geograficamente onde a operação falha, cruzando estado e trimestre. Calcula SLA (lead time médio e % no prazo) por estado e por trimestre separadamente, depois cruza os dois no heatmap.
>
> 1. **Lead time médio por estado:** Ranking horizontal dos 20 estados com maior lead time. Barras vermelhas marcam os estados no quartil superior — os gargalos geográficos primários. A linha tracejada mostra a mediana nacional como referência.
> 2. **% no prazo por estado (piores):** Ordena os estados pelo pior SLA e colore por nível de gravidade — vermelho abaixo de 80%, laranja entre 80% e 90%, verde acima. Estados que aparecem em vermelho nos dois gráficos são os gargalos críticos confirmados.
> 3. **Lead time e % no prazo por trimestre:** Gráfico de duplo eixo que mostra se a degradação de SLA é consistente ao longo do tempo ou concentrada em períodos específicos — diferencia gargalo estrutural de gargalo sazonal.
> 4. **Heatmap estado × trimestre:** A visualização mais densa do bloco. Cada célula mostra o % de entregas no prazo para aquele estado naquele trimestre. Células vermelhas persistentes em todos os trimestres confirmam gargalo estrutural. Células vermelhas concentradas em um trimestre específico indicam problema sazonal ou pontual.

**Análise do Resultado:** 
Aplicando a Teoria das Restrições, localizamos onde o sistema "estoura" primeiro. O mapa de calor (Heatmap) revela se os atrasos são geográficos (certos estados colapsam sempre) ou sazonais (o sistema inteiro sofre em épocas como Black Friday). Se o gargalo for estrutural (mesmos estados vermelhos o ano todo), a solução é estratégica (novos parceiros locais). Se for sazonal, o problema é de planejamento de demanda e capacidade temporária.

In [ ]:
sla_estado = (
    fe.groupby("estado_cliente")
    .agg(
        n_pedidos    = ("id_pedido",         "nunique"),
        lead_medio   = ("lead_time_dias",    "mean"),
        pct_no_prazo = ("entregue_no_prazo", "mean"),
        nota_media   = ("nota_review",       "mean"),
    )
    .reset_index().sort_values("lead_medio", ascending=False)
)
sla_estado["pct_no_prazo"] = pd.to_numeric(sla_estado["pct_no_prazo"], errors="coerce") * 100

sla_periodo = (
    fe.groupby("periodo")
    .agg(
        n_pedidos    = ("id_pedido",         "nunique"),
        lead_medio   = ("lead_time_dias",    "mean"),
        pct_no_prazo = ("entregue_no_prazo", "mean"),
    )
    .reset_index().sort_values("periodo")
)
sla_periodo["pct_no_prazo"] = pd.to_numeric(sla_periodo["pct_no_prazo"], errors="coerce") * 100

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("Análise 4 — Gargalos por Estado e Período (Teoria das Restrições)",
             fontsize=13, fontweight="bold")

top_est = sla_estado.head(20).sort_values("lead_medio", ascending=True)
cores_lt = [COR_ALERTA if v > sla_estado["lead_medio"].quantile(0.75)
            else COR_MARGEM if v < sla_estado["lead_medio"].quantile(0.25)
            else COR_NEUTRO for v in top_est["lead_medio"]]
axes[0,0].barh(top_est["estado_cliente"], top_est["lead_medio"], color=cores_lt, alpha=0.85)
axes[0,0].axvline(sla_estado["lead_medio"].median(), color=COR_DESTAQUE, linestyle="--", linewidth=1.5,
                  label=f"Mediana: {sla_estado['lead_medio'].median():.1f}d")
axes[0,0].set_xlabel("Lead Time Médio (dias)")
axes[0,0].set_title("Lead Time Médio por Estado", fontsize=11)
axes[0,0].legend(frameon=False, fontsize=8)

top_est_prazo = sla_estado.sort_values("pct_no_prazo", ascending=True).head(20)
cores_prazo = [COR_ALERTA if v < 80 else COR_DESTAQUE if v < 90 else COR_MARGEM for v in top_est_prazo["pct_no_prazo"]]
axes[0,1].barh(top_est_prazo["estado_cliente"], top_est_prazo["pct_no_prazo"], color=cores_prazo, alpha=0.85)
axes[0,1].axvline(90, color=COR_NEUTRO, linestyle="--", linewidth=1, alpha=0.7, label="SLA 90%")
axes[0,1].set_xlabel("% Entregue no Prazo")
axes[0,1].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0,1].set_title("% no Prazo por Estado (piores)", fontsize=11)
axes[0,1].legend(frameon=False, fontsize=8)

xq = range(len(sla_periodo))
axes[1,0].bar(xq, sla_periodo["lead_medio"], color=COR_RECEITA, alpha=0.7, label="Lead médio")
ax3_twin = axes[1,0].twinx()
ax3_twin.plot(xq, sla_periodo["pct_no_prazo"], color=COR_MARGEM,
              linewidth=2, marker="o", markersize=5, label="% no prazo")
ax3_twin.set_ylabel("% no Prazo", color=COR_MARGEM)
ax3_twin.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1,0].set_ylabel("Lead Time Médio (dias)")
axes[1,0].set_title("Lead Time e % no Prazo por Trimestre", fontsize=11)
axes[1,0].set_xticks(xq)
axes[1,0].set_xticklabels(sla_periodo["periodo"], rotation=45, ha="right", fontsize=8)
lines1, labels1 = axes[1,0].get_legend_handles_labels()
lines2, labels2 = ax3_twin.get_legend_handles_labels()
axes[1,0].legend(lines1+lines2, labels1+labels2, frameon=False, fontsize=8)

top15_est = sla_estado.nlargest(15, "n_pedidos")["estado_cliente"].tolist()
hm_data = (
    fe[fe["estado_cliente"].isin(top15_est)]
    .groupby(["estado_cliente", "periodo"])["entregue_no_prazo"].mean().mul(100)
    .reset_index()
    .pivot(index="estado_cliente", columns="periodo", values="entregue_no_prazo")
    .astype(float)  # garante dtype float — evita TypeError no seaborn heatmap
)
sns.heatmap(
    hm_data, ax=axes[1,1], cmap="RdYlGn", vmin=60, vmax=100,
    annot=True, fmt=".0f", linewidths=0.4,
    cbar_kws={"label": "% no Prazo"}, annot_kws={"size": 7}
)
axes[1,1].set_title("% Entregas no Prazo — Top 15 Estados × Trimestre", fontsize=11)
axes[1,1].tick_params(axis="x", labelsize=7, rotation=45)
axes[1,1].tick_params(axis="y", labelsize=8)

plt.tight_layout()
salvar(fig, "04_sla_estado_periodo")
plt.show()

piores_est = sla_estado.nsmallest(5, "pct_no_prazo")[["estado_cliente","pct_no_prazo","lead_medio","n_pedidos"]]
pior_trim  = sla_periodo.loc[sla_periodo["pct_no_prazo"].idxmin()]
print("\n5 estados com pior SLA:")
for _, r in piores_est.iterrows():
    print(f"  {r['estado_cliente']}: {r['pct_no_prazo']:.1f}% no prazo | lead: {r['lead_medio']:.1f}d | {r['n_pedidos']} pedidos")
print(f"\nPior trimestre: {pior_trim['periodo']} — {pior_trim['pct_no_prazo']:.1f}% no prazo")

---

## Análise 5 — Em que ponto o crescimento passa a gerar ineficiência operacional?

> *"A determinação do volume exato a partir do qual a eficiência operacional começa a degradar fornece um dado acionável para o planejamento estratégico. Através de regressão segmentada, identifica-se o ponto de inflexão onde o crescimento deixa de gerar diluição de custos e passa a consumir a qualidade do serviço. Este número define a capacidade máxima sustentável da estrutura atual antes da necessidade de novos aportes.

**Framework:** Matriz GUT + Análise de Tendência (PDCA) — ponto de ruptura da capacidade  
**Entrega:** Regressão segmentada que identifica o volume a partir do qual o lead time degrada

**Como este script responde à pergunta:**
> Este script usa uma técnica de regressão segmentada para encontrar automaticamente o volume a partir do qual o lead time começa a crescer de forma sistemática. Testa todos os pontos de corte possíveis na série de volume × lead time e identifica qual divisão produz as duas retas com melhor ajuste combinado — esse ponto é o limiar de capacidade operacional.
>
> 1. **Scatter volume × lead time com regressão segmentada:** Cada ponto é um mês, colorido pelo % de entregas no prazo. Dois segmentos de reta são ajustados: um antes e um depois do ponto de inflexão. A linha verde (antes do limiar) mostra o comportamento da operação dentro da capacidade; a linha vermelha (depois do limiar) mostra o comportamento sob sobrecarga. O ponto de corte é anotado diretamente no gráfico com o volume correspondente.
> 2. **Scatter volume × % no prazo com regressão segmentada:** O mesmo recorte aplicado ao SLA. Se a reta após o limiar for descendente, confirma que acima daquele volume a taxa de entregas no prazo cai sistematicamente — o sistema tem um teto de capacidade mensurável e identificado.

**Análise do Resultado:**
 Esta é a "pergunta de um milhão de dólares" para o investidor. Através da regressão segmentada, buscamos o ponto de ruptura: o volume exato a partir do qual a eficiência cai. Abaixo desse ponto, a empresa é um ótimo investimento de crescimento. Acima dele, cada novo pedido custa mais caro e demora mais para chegar. Saber esse número permite ao comprador planejar o gatilho de intervenção: abaixo do limiar, crescer é seguro; acima, qualquer aceleração de volume exigirá mudança operacional antes de comprometer o SLA


In [ ]:
df_inflexao = lt_mensal[["n_pedidos", "lead_medio", "pct_no_prazo", "ano_mes"]].dropna().sort_values("n_pedidos")

def regressao_segmentada(x_vals, y_vals):
    melhor_r2, melhor_split = -np.inf, None
    for i in range(2, len(x_vals) - 2):
        x1, y1 = x_vals[:i], y_vals[:i]
        x2, y2 = x_vals[i:], y_vals[i:]
        if len(x1) < 3 or len(x2) < 3: continue
        r1 = stats.pearsonr(x1, y1)[0] ** 2
        r2 = stats.pearsonr(x2, y2)[0] ** 2
        r2_combo = (r1 * len(x1) + r2 * len(x2)) / (len(x1) + len(x2))
        if r2_combo > melhor_r2:
            melhor_r2, melhor_split = r2_combo, i
    return melhor_split

x_arr = df_inflexao["n_pedidos"].values
y_arr = df_inflexao["lead_medio"].values
split_idx    = regressao_segmentada(x_arr, y_arr)
vol_inflexao = x_arr[split_idx] if split_idx else None

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Análise 5 — Ponto de Inflexão: Onde o Crescimento Vira Ineficiência", fontsize=13, fontweight="bold")

axes[0].scatter(
    df_inflexao["n_pedidos"], df_inflexao["lead_medio"],
    c=df_inflexao["pct_no_prazo"], cmap="RdYlGn", vmin=70, vmax=100,
    s=60, alpha=0.8, edgecolors="white", linewidth=0.5, zorder=5
)
for _, row in df_inflexao.iterrows():
    axes[0].annotate(row["ano_mes"][2:], (row["n_pedidos"], row["lead_medio"]),
                     fontsize=5, alpha=0.5, xytext=(2, 2), textcoords="offset points")

if split_idx and split_idx > 2:
    m1, b1, _, _, _ = stats.linregress(x_arr[:split_idx], y_arr[:split_idx])
    m2, b2, _, _, _ = stats.linregress(x_arr[split_idx:], y_arr[split_idx:])
    xf1 = np.linspace(x_arr[0], x_arr[split_idx-1], 50)
    xf2 = np.linspace(x_arr[split_idx], x_arr[-1], 50)
    axes[0].plot(xf1, m1*xf1+b1, color=COR_MARGEM,  linewidth=2, label=f"Até {vol_inflexao:,.0f}")
    axes[0].plot(xf2, m2*xf2+b2, color=COR_ALERTA,  linewidth=2, label=f"Acima de {vol_inflexao:,.0f}")
    axes[0].axvline(vol_inflexao, color="black", linewidth=1.5, linestyle=":", alpha=0.7)
    axes[0].text(
        vol_inflexao * 1.01, axes[0].get_ylim()[1] * 0.97,
        f"Inflexão:\n~{vol_inflexao:,.0f} ped/mês",
        fontsize=8, color="black",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8)
    )

axes[0].set_xlabel("Volume Mensal (pedidos)")
axes[0].set_ylabel("Lead Time Médio (dias)")
axes[0].set_title("Ponto de Inflexão: Volume × Lead Time", fontsize=11)
axes[0].legend(frameon=False, fontsize=8)

axes[1].scatter(df_inflexao["n_pedidos"], df_inflexao["pct_no_prazo"],
                color=COR_RECEITA, s=60, alpha=0.8, edgecolors="white", linewidth=0.5)
if split_idx and split_idx > 2:
    y_prazo = df_inflexao["pct_no_prazo"].values
    mp1, bp1, _, _, _ = stats.linregress(x_arr[:split_idx], y_prazo[:split_idx])
    mp2, bp2, _, _, _ = stats.linregress(x_arr[split_idx:], y_prazo[split_idx:])
    axes[1].plot(xf1, mp1*xf1+bp1, color=COR_MARGEM, linewidth=2)
    axes[1].plot(xf2, mp2*xf2+bp2, color=COR_ALERTA, linewidth=2)
    axes[1].axvline(vol_inflexao, color="black", linewidth=1.5, linestyle=":", alpha=0.7)
axes[1].axhline(90, color=COR_NEUTRO, linestyle="--", linewidth=1, alpha=0.7, label="SLA 90%")
axes[1].set_xlabel("Volume Mensal (pedidos)")
axes[1].set_ylabel("% Entregue no Prazo")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_title("Ponto de Inflexão: Volume × % no Prazo", fontsize=11)
axes[1].legend(frameon=False, fontsize=8)

plt.tight_layout()
salvar(fig, "05_ponto_inflexao")
plt.show()

if vol_inflexao:
    lt_antes  = y_arr[:split_idx].mean()
    lt_depois = y_arr[split_idx:].mean()
    delta_lt  = lt_depois - lt_antes
    print(f"\nInflexão: ~{vol_inflexao:,.0f} ped/mês")
    print(f"Lead antes: {lt_antes:.1f}d | Lead depois: {lt_depois:.1f}d | Degradação: +{delta_lt:.1f}d ({delta_lt/lt_antes*100:.0f}%)")
else:
    delta_lt = 0
    print("Inflexão não detectada.")

---

## Análise 6 — Os gargalos operacionais são estruturais ou sazonais?

> *"Restrições estruturais exigem mudanças na capacidade operacional instalada, enquanto gargalos sazonais demandam planejamento de pico. A correta classificação da natureza do gargalo define o tipo e a urgência das intervenções necessárias no pós-aquisição"*

**Framework:** PDCA — distinção entre problema estrutural e problema de gestão de pico  
**Entrega:** Distribuição do lead time por mês do ano + % no prazo sazonal, identificando meses críticos e se os gargalos coincidem com picos de volume

**Como este script responde à pergunta:**
> Um gargalo que aparece em novembro é diferente de um gargalo que existe o ano todo — o tratamento e o investimento necessário são completamente diferentes. O script calcula o lead time e o SLA para cada mês do calendário, agregando todos os anos disponíveis, e compara com o volume médio daquele mês para ver se os piores meses coincidem com os de maior demanda.
>
> 1. **Boxplot de lead time por mês:** Mostra a distribuição completa (mediana, quartis e outliers) do lead time para cada mês do ano. Meses com caixas deslocadas para cima e mais largas têm lead time sistemicamente maior e mais variável. A coloração destaca os meses no quartil superior de mediana — os meses estruturalmente mais lentos.
> 2. **% no prazo e volume por mês:** Barras mostram o SLA de cada mês; a linha roxa sobreposta mostra o volume médio. Se as barras vermelhas (SLA baixo) coincidirem com os picos da linha de volume, o gargalo é sazonal — resolvível com planejamento de pico. Se as barras vermelhas estiverem distribuídas independentemente do volume, o problema é estrutural e exige intervenção na capacidade instalada.

**Análise do Resultado:** Esta análise diferencia um "problema de momento" de um "problema de fundação". Se o gargalo (atraso ou custo alto) aparece apenas em meses específicos, como a Black Friday, o problema é sazonal e pode ser resolvido com planejamento temporário. Porém, se os mesmos estados ou categorias apresentam falhas de forma persistente em todos os trimestres, o gargalo é estrutural. Para um investidor, gargalos estruturais exigem mudanças profundas na malha logística ou troca de fornecedores, enquanto os sazonais exigem apenas melhor gestão de estoque e demanda.


In [ ]:
MESES_ORD = ["Janeiro","Fevereiro","Março","Abril","Maio","Junho",
             "Julho","Agosto","Setembro","Outubro","Novembro","Dezembro"]

sazonal_sla = (
    fe.groupby(["mes", "nome_mes"])
    .agg(
        lead_medio   = ("lead_time_dias",    "mean"),
        pct_no_prazo = ("entregue_no_prazo", "mean"),
        n_pedidos    = ("id_pedido",         "nunique"),
        nota_media   = ("nota_review",       "mean"),
    )
    .reset_index().sort_values("mes")
)
sazonal_sla["pct_no_prazo"] = pd.to_numeric(sazonal_sla["pct_no_prazo"], errors="coerce") * 100
fe["nome_mes_ord"] = pd.Categorical(fe["nome_mes"], categories=MESES_ORD, ordered=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Análise 6 — Os Gargalos são Estruturais ou Sazonais?", fontsize=13, fontweight="bold")

# Boxplot de lead time por mês
meses_presentes = [m for m in MESES_ORD if m in fe["nome_mes_ord"].values]
fe_box = fe[fe["lead_time_dias"] <= fe["lead_time_dias"].quantile(0.98)]
dados_box = [fe_box[fe_box["nome_mes_ord"] == m]["lead_time_dias"].dropna().values
             for m in meses_presentes]
bp = axes[0].boxplot(dados_box, patch_artist=True, widths=0.6,
                     medianprops=dict(color="white", linewidth=2),
                     whiskerprops=dict(linewidth=1),
                     flierprops=dict(marker=".", markersize=2, alpha=0.3))
medians = [np.median(d) if len(d) > 0 else 0 for d in dados_box]
threshold_alto = np.percentile(medians, 75)
for patch, med in zip(bp["boxes"], medians):
    patch.set_facecolor(COR_ALERTA if med > threshold_alto else COR_RECEITA)
    patch.set_alpha(0.7)
axes[0].set_xticklabels([m[:3] for m in meses_presentes], rotation=0, fontsize=9)
axes[0].set_ylabel("Lead Time (dias)")
axes[0].set_title("Distribuição do Lead Time por Mês", fontsize=11)
axes[0].axhline(fe["lead_time_dias"].median(), color=COR_NEUTRO, linestyle="--",
                linewidth=1, alpha=0.7, label=f"Mediana geral: {fe['lead_time_dias'].median():.0f}d")
axes[0].legend(frameon=False, fontsize=8)

# % no prazo × volume por mês
x_m = range(len(sazonal_sla))
axes[1].bar(x_m, sazonal_sla["pct_no_prazo"],
            color=[COR_ALERTA if v < 85 else COR_MARGEM for v in sazonal_sla["pct_no_prazo"]], alpha=0.85)
ax_twin = axes[1].twinx()
ax_twin.plot(x_m, sazonal_sla["n_pedidos"], color=COR_ROXO,
             linewidth=2, marker="o", markersize=4, label="Volume")
ax_twin.set_ylabel("Nº Pedidos", color=COR_ROXO)
axes[1].set_xticks(x_m)
axes[1].set_xticklabels([m[:3] for m in sazonal_sla["nome_mes"]], rotation=0, fontsize=9)
axes[1].set_ylabel("% Entregue no Prazo")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].axhline(90, color=COR_NEUTRO, linestyle="--", linewidth=1, alpha=0.7, label="SLA 90%")
axes[1].set_title("% no Prazo e Volume por Mês do Ano", fontsize=11)
axes[1].legend(frameon=False, fontsize=8)

plt.tight_layout()
salvar(fig, "06_sazonalidade_sla")
plt.show()

pior_mes_sla   = sazonal_sla.loc[sazonal_sla["pct_no_prazo"].idxmin()]
pior_mes_volume = sazonal_sla.loc[sazonal_sla["n_pedidos"].idxmax()]
variacao_sazonal = sazonal_sla["pct_no_prazo"].max() - sazonal_sla["pct_no_prazo"].min()
gargalo_sazonal = pior_mes_sla["nome_mes"] == pior_mes_volume["nome_mes"]

print("\n" + "="*60)
print("INSIGHT — SAZONALIDADE DO SLA")
print("="*60)
print(f"Pior mês para SLA    : {pior_mes_sla['nome_mes']} — {pior_mes_sla['pct_no_prazo']:.1f}% no prazo")
print(f"Mês de maior volume  : {pior_mes_volume['nome_mes']} — {pior_mes_volume['n_pedidos']} pedidos")
print(f"Variação sazonal     : {variacao_sazonal:.1f}pp ({'alta' if variacao_sazonal > 15 else 'moderada' if variacao_sazonal > 8 else 'baixa'})")
print(f"Gargalo sazonal      : {'SIM — pico de volume coincide com pior SLA → gargalo de capacidade' if gargalo_sazonal else 'NÃO — pior SLA não coincide com pico de volume → gargalo estrutural'}")

---

## Análise 7 — O atraso está na preparação do pedido ou na entrega final?

> *"A decomposição do lead time total em etapas (preparação interna vs. transporte externo) localiza a responsabilidade pelos atrasos. Identificar se o gargalo reside no processamento do pedido (controlável internamente) ou no percurso final (dependente de terceiros) é essencial para definir o plano de ação: melhoria de processos internos ou renegociação de contratos logísticos."*

**Framework:** Análise de fluxo de processo   
**Entrega:** Decomposição do lead time em tempo de preparação e tempo de transporte, com evolução temporal e comparação por estado

**Como este script responde à pergunta:**
> O lead time total de um pedido tem dois responsáveis distintos: o seller/Olist (da aprovação do pagamento até o envio para a transportadora) e a transportadora (do envio até a entrega ao cliente). Saber qual dos dois está consumindo mais tempo muda completamente a solução — um é um problema interno, o outro é um problema de parceiro logístico. O script calcula os dois tempos separadamente para cada pedido e agrega mensalmente.
>
> 1. **Barras empilhadas preparação + transporte:** Cada barra é um mês com os dois componentes empilhados em cores distintas. Se a fatia azul (preparação) cresce ao longo do tempo, o gargalo está na operação interna. Se a fatia laranja (transporte) cresce, o problema está na rede logística.
> 2. **% de cada etapa no lead time total:** As duas linhas mostram como a participação de cada componente evolui. Uma linha de preparação crescendo acima de 50% e em tendência de alta é o diagnóstico mais direto de que o problema de SLA está dentro de casa — e é endereçável sem depender de terceiros.
>
> **Nota:** esta análise só executa se as colunas `data_aprovacao` e `data_envio_transportadora` estiverem disponíveis no dataset. Caso contrário, uma mensagem de aviso é exibida e o notebook continua normalmente.

**Análise do Resultado:** 
Aqui isolamos a responsabilidade para saber onde "estancar o sangue". Se o atraso ocorre na preparação (dentro do armazém/seller), o problema é de gestão interna e processos. Se o atraso ocorre no transporte, o problema é da transportadora ou da infraestrutura das estradas. Identificar se o erro é "dentro de casa" ou "na rua" é fundamental para decidir se o investimento pós-aquisição deve ir para tecnologia de separação de pedidos (WMS) ou para a contratação de novos parceiros logísticos.

In [ ]:
if not tem_decomposicao:
    print("[AVISO] Colunas data_aprovacao ou data_envio_transportadora insuficientes.")
    print("A decomposição do lead time não está disponível para este dataset.")
    print("Para ativar: verifique se o ETL exportou essas colunas corretamente.")
else:
    decomp_mensal = (
        fe.groupby("ano_mes")
        .agg(
            preparacao_medio  = ("tempo_preparacao", "mean"),
            transporte_medio  = ("tempo_transporte",  "mean"),
            lead_total_medio  = ("lead_time_dias",    "mean"),
        )
        .reset_index().sort_values("ano_mes")
    )
    decomp_mensal["pct_preparacao"] = decomp_mensal["preparacao_medio"] / decomp_mensal["lead_total_medio"] * 100
    decomp_mensal["pct_transporte"] = decomp_mensal["transporte_medio"] / decomp_mensal["lead_total_medio"] * 100

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Análise 7 — Decomposição do Lead Time: Preparação vs Transporte",
                 fontsize=13, fontweight="bold")

    x     = range(len(decomp_mensal))
    xtick = list(range(0, len(decomp_mensal), 3))
    xlabs = [decomp_mensal["ano_mes"].iloc[i] for i in xtick]

    # Barras empilhadas: preparação + transporte
    axes[0].bar(x, decomp_mensal["preparacao_medio"], color=COR_RECEITA, alpha=0.85, label="Preparação (seller/Olist)")
    axes[0].bar(x, decomp_mensal["transporte_medio"],
                bottom=decomp_mensal["preparacao_medio"],
                color=COR_DESTAQUE, alpha=0.85, label="Transporte (transportadora)")
    axes[0].set_title("Lead Time Médio: Preparação vs Transporte (dias)", fontsize=11)
    axes[0].set_ylabel("Dias")
    axes[0].set_xticks(xtick); axes[0].set_xticklabels(xlabs, rotation=45, ha="right", fontsize=8)
    axes[0].legend(frameon=False, fontsize=8)

    # % de cada etapa
    axes[1].plot(x, decomp_mensal["pct_preparacao"], color=COR_RECEITA,
                 linewidth=2, marker="o", markersize=4, label="% Preparação")
    axes[1].plot(x, decomp_mensal["pct_transporte"], color=COR_DESTAQUE,
                 linewidth=2, marker="s", markersize=4, label="% Transporte")
    axes[1].axhline(50, color=COR_NEUTRO, linestyle="--", linewidth=1, alpha=0.5)
    axes[1].set_title("Participação de Cada Etapa no Lead Time Total", fontsize=11)
    axes[1].set_ylabel("%")
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
    axes[1].set_xticks(xtick); axes[1].set_xticklabels(xlabs, rotation=45, ha="right", fontsize=8)
    axes[1].legend(frameon=False, fontsize=8)

    plt.tight_layout()
    salvar(fig, "07_decomposicao_lead_time")
    plt.show()

    prep_media = fe["tempo_preparacao"].mean()
    trans_media = fe["tempo_transporte"].mean()
    total_media = fe["lead_time_dias"].mean()
    gargalo_principal = "PREPARAÇÃO (seller/Olist)" if prep_media > trans_media else "TRANSPORTE (transportadora)"

    print("\n" + "="*60)
    print("INSIGHT — DECOMPOSIÇÃO DO LEAD TIME")
    print("="*60)
    print(f"Lead time total médio      : {total_media:.1f} dias")
    print(f"Tempo de preparação médio  : {prep_media:.1f} dias ({prep_media/total_media*100:.0f}% do total)")
    print(f"Tempo de transporte médio  : {trans_media:.1f} dias ({trans_media/total_media*100:.0f}% do total)")
    print(f"Gargalo principal          : {gargalo_principal}")
    print(f"\nImplicação: a solução prioritária é {'melhorar o processo interno de separação/envio' if prep_media > trans_media else 'negociar/substituir parceiros logísticos'}")

---

## Análise 8 — A degradação operacional em períodos de pico afeta todas as regiões igualmente — ou há regiões que colapsam antes das outras?

> *"Médias nacionais frequentemente mascaram colapsos operacionais em regiões específicas. Uma taxa de entrega satisfatória no agregado pode ocultar estados com níveis de serviço críticos. Analisar a degradação do SLA de forma geolocalizada permite identificar vulnerabilidades regionais que não aparecem no consolidado nacional — informação crítica para o comprador avaliar o risco operacional real do ativo."*

**Framework:** Análise de variância geoespacial sob pressão (stress testing regional)  
**Entrega:** Heatmap de SLA por estado × trimestre + dispersão da degradação regional em picos

**Como este script responde à pergunta:**
> A análise regional de degradação é o que separa um problema operacional gerenciável de um risco estrutural concentrado. Este script constrói duas visualizações:
>
> 1. **Heatmap SLA por estado × trimestre:** Cada célula mostra o % de pedidos entregues no prazo em determinado estado e trimestre. A escala de cor vai do vermelho (SLA crítico, abaixo de 70%) ao verde (SLA excelente, acima de 90%). O padrão que mais preocupa um comprador é ver estados específicos ficando progressivamente mais vermelhos ao longo dos trimestres — isso indica colapso regional estrutural, não variação pontual. Se o heatmap é uniformemente verde com manchas espalhadas, o problema é operacional geral; se estados específicos sistematicamente aparecem em vermelho, é um gargalo geográfico com causa identificável.
> 2. **Dispersão da degradação em trimestres de pico:** Identifica os trimestres de maior volume (top 3) e calcula para cada estado quanto o SLA caiu nesses períodos em relação à sua própria média histórica. Estados com queda acima de 10pp em picos são os que colapsam primeiro — e são os que precisam de investimento logístico prioritário no plano pós-aquisição. O tamanho de cada bolha representa o volume de pedidos do estado, permitindo identificar se os estados que colapsam são relevantes ou marginais para a operação.

**Análise do Resultado:**
 Esta análise mede a "fragilidade regional". É comum que a operação funcione bem no Sudeste, mas entre em colapso total no Norte ou Nordeste durante picos de venda. Identificamos quais regiões são as primeiras a "quebrar" quando o volume sobe. Para o comprador, isso revela o risco geográfico: se a tese de crescimento da empresa depende de expandir para o Nordeste, mas os dados mostram que a operação já colapsa lá com volumes baixos, o plano de expansão precisará ser totalmente revisado.


In [ ]:
# ─── Análise 8 — Degradação Regional em Períodos de Pico ─────────────────────

# SLA por estado e trimestre
sla_reg = (
    fe.groupby(["estado_cliente", "periodo"])
    .agg(
        pct_no_prazo = ("entregue_no_prazo", "mean"),
        n_pedidos    = ("id_pedido",         "nunique"),
    )
    .reset_index()
)
sla_reg["pct_no_prazo"] = pd.to_numeric(sla_reg["pct_no_prazo"], errors="coerce") * 100

# Filtrar estados com volume mínimo (pelo menos 30 pedidos por trimestre em média)
estados_validos = (
    sla_reg.groupby("estado_cliente")["n_pedidos"].mean()
    .pipe(lambda s: s[s >= 30].index.tolist())
)
sla_reg = sla_reg[sla_reg["estado_cliente"].isin(estados_validos)]

# Pivot para heatmap
periodos_ord = sorted(sla_reg["periodo"].unique())
sla_pivot = sla_reg.pivot_table(
    index="estado_cliente", columns="periodo",
    values="pct_no_prazo"
)
# Ordenar estados por SLA médio (piores primeiro)
sla_pivot = sla_pivot.reindex(sla_pivot.mean(axis=1).sort_values().index)

# Trimestres de pico (top 3 por volume)
vol_por_trim = fe.groupby("periodo")["id_pedido"].nunique().sort_values(ascending=False)
top3_pico = vol_por_trim.head(3).index.tolist()

# SLA médio histórico por estado
sla_medio_estado = sla_reg.groupby("estado_cliente")["pct_no_prazo"].mean()
vol_total_estado = sla_reg.groupby("estado_cliente")["n_pedidos"].sum()

# Degradação em picos
sla_pico = (
    sla_reg[sla_reg["periodo"].isin(top3_pico)]
    .groupby("estado_cliente")["pct_no_prazo"].mean()
)
df_deg = pd.DataFrame({
    "sla_medio": sla_medio_estado,
    "sla_pico":  sla_pico,
    "n_pedidos": vol_total_estado,
}).dropna()
df_deg["degradacao_pp"] = df_deg["sla_pico"] - df_deg["sla_medio"]
df_deg = df_deg[df_deg.index.isin(estados_validos)].sort_values("degradacao_pp")

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
fig.suptitle("Análise 8 — Degradação Operacional Regional em Períodos de Pico", fontsize=13, fontweight="bold")

# Heatmap SLA por estado × trimestre
import matplotlib.colors as mcolors
cmap_sla = plt.cm.RdYlGn
im = axes[0].imshow(
    sla_pivot.values, aspect="auto", cmap=cmap_sla, vmin=60, vmax=100
)
plt.colorbar(im, ax=axes[0], label="% no prazo", fraction=0.03)
axes[0].set_xticks(range(len(sla_pivot.columns)))
axes[0].set_xticklabels([str(p) for p in sla_pivot.columns], rotation=45, ha="right", fontsize=8)
axes[0].set_yticks(range(len(sla_pivot.index)))
axes[0].set_yticklabels(sla_pivot.index, fontsize=9)
axes[0].set_title("SLA (% no prazo) por Estado × Trimestre\n(vermelho = colapso | verde = excelente)", fontsize=11)
for r in range(len(sla_pivot.index)):
    for c in range(len(sla_pivot.columns)):
        val = sla_pivot.values[r, c]
        if not np.isnan(val):
            axes[0].text(c, r, f"{val:.0f}%", ha="center", va="center",
                         fontsize=7, color="white" if val < 72 else "black")

# Scatter de degradação em picos
cores_deg = [COR_ALERTA if v < -10 else COR_DESTAQUE if v < -5 else COR_MARGEM
             for v in df_deg["degradacao_pp"]]
sc = axes[1].scatter(
    df_deg["sla_medio"],
    df_deg["degradacao_pp"],
    s=df_deg["n_pedidos"] / df_deg["n_pedidos"].max() * 600 + 40,
    c=cores_deg, alpha=0.8, edgecolors="white", linewidth=0.5, zorder=5
)
for estado, row in df_deg.iterrows():
    axes[1].annotate(
        estado,
        (row["sla_medio"], row["degradacao_pp"]),
        fontsize=8, ha="center", xytext=(0, 6), textcoords="offset points"
    )
axes[1].axhline(-10, color=COR_ALERTA, linestyle="--", linewidth=1, alpha=0.8, label="Limiar crítico (−10pp)")
axes[1].axhline(-5,  color=COR_DESTAQUE, linestyle=":", linewidth=1, alpha=0.7, label="Atenção (−5pp)")
axes[1].axhline(0,   color="black",     linestyle="-",  linewidth=0.8, alpha=0.4)
axes[1].set_xlabel("SLA médio histórico do estado (%)")
axes[1].set_ylabel("Variação do SLA em trimestres de pico (pp)")
axes[1].set_title(f"Degradação Regional nos Top 3 Trimestres de Pico\n(tamanho = volume de pedidos)", fontsize=11)
axes[1].legend(frameon=False, fontsize=9)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:+.0f}pp"))

plt.tight_layout()
salvar(fig, "08_degradacao_regional")
plt.show()

# Métricas
estados_colapso = df_deg[df_deg["degradacao_pp"] < -10]
estado_mais_fragil = df_deg.sort_values("degradacao_pp").iloc[0]

print("\n" + "="*55)
print("INSIGHT — DEGRADAÇÃO REGIONAL EM PICOS")
print("="*55)
print(f"Estados com colapso em pico (queda >10pp): {len(estados_colapso)}")
if len(estados_colapso):
    print(f"  Estados críticos: {', '.join(estados_colapso.index.tolist())}")
print(f"Estado mais frágil : {estado_mais_fragil.name} "
      f"(SLA médio: {estado_mais_fragil['sla_medio']:.1f}% | queda em pico: {estado_mais_fragil['degradacao_pp']:+.1f}pp)")
tipo_problema = "geográfico concentrado" if len(estados_colapso) <= 3 else "sistêmico generalizado"
print(f"Tipo de problema logístico : {tipo_problema}")


---

## Síntese do Bloco 3 — Escalabilidade Operacional
> **Limitações desta análise:** a análise de custo unitário utiliza frete como proxy de custo variável — custos operacionais internos (armazenagem, handling, tecnologia) não estão disponíveis no dataset. O ponto de inflexão calculado por regressão segmentada é uma estimativa estatística baseada no período analisado, não um limite físico de capacidade. A decomposição do lead time assume que os timestamps de envio e entrega no dataset refletem fielmente a operação real.


In [ ]:
# ─── Veredicto 100% dinâmico — recalcula tudo localmente ────────────────────
# Custo unitário de frete
_frete_mensal = (
    fe.groupby("ano_mes")
    .agg(n_pedidos=("id_pedido","nunique"), frete_medio=("valor_frete","mean"),
         frete_total=("valor_frete","sum"), receita_total=("preco","sum"))
    .reset_index().sort_values("ano_mes")
)
_frete_mensal["pct_frete"] = _frete_mensal["frete_total"] / _frete_mensal["receita_total"] * 100
_r_vol_frete, _ = stats.pearsonr(_frete_mensal["n_pedidos"], _frete_mensal["frete_medio"])
_frete_medio_geral  = fe["valor_frete"].mean()
_pct_frete_receita  = fe["valor_frete"].sum() / fe["preco"].sum() * 100

# Lead time
_lt_mensal = (
    fe.groupby("ano_mes")
    .agg(n_pedidos=("id_pedido","nunique"),
         lead_medio=("lead_time_dias","mean"),
         pct_no_prazo=("entregue_no_prazo","mean"))
    .reset_index().sort_values("ano_mes")
)
_lt_mensal["pct_no_prazo"] = pd.to_numeric(_lt_mensal["pct_no_prazo"], errors="coerce") * 100
_mask_lt = _lt_mensal["lead_medio"].notna() & _lt_mensal["n_pedidos"].notna()
_r_lt, _p_lt = stats.pearsonr(
    _lt_mensal.loc[_mask_lt, "n_pedidos"],
    _lt_mensal.loc[_mask_lt, "lead_medio"]
)
_lead_medio_geral = fe["lead_time_dias"].mean()
_lead_p90         = fe["lead_time_dias"].quantile(0.9)
_pct_no_prazo     = pd.to_numeric(fe["entregue_no_prazo"], errors="coerce").mean() * 100

# Falhas operacionais
_falhas = (
    fato.groupby("ano_mes")
    .agg(n_total=("id_pedido","nunique"),
         n_canc=("status_pedido", lambda x: (x=="cancelado").sum()))
    .reset_index().sort_values("ano_mes")
)
_atrasos = fe[fe["atraso_dias"]>0].groupby("ano_mes").agg(n_atr=("id_pedido","nunique")).reset_index()
_falhas  = _falhas.merge(_atrasos, on="ano_mes", how="left").fillna({"n_atr": 0})
_falhas["pct_canc"] = _falhas["n_canc"] / _falhas["n_total"] * 100
_falhas["pct_atr"]  = _falhas["n_atr"]  / _falhas["n_total"] * 100
_r_vol_canc, _p_canc = stats.pearsonr(_falhas["n_total"], _falhas["pct_canc"])
_r_vol_atr,  _p_atr  = stats.pearsonr(_falhas["n_total"], _falhas["pct_atr"])
_taxa_canc = _falhas["pct_canc"].mean()
_taxa_atr  = _falhas["pct_atr"].mean()

# SLA por estado
_sla_est = (
    fe.groupby("estado_cliente")
    .agg(pct_no_prazo=("entregue_no_prazo","mean"), n_pedidos=("id_pedido","nunique"))
    .reset_index()
)
_sla_est["pct_no_prazo"] = pd.to_numeric(_sla_est["pct_no_prazo"], errors="coerce") * 100
_piores_est  = _sla_est.sort_values("pct_no_prazo").head(1)
_pior_estado = _piores_est.iloc[0]["estado_cliente"]
_pior_sla_est = _piores_est.iloc[0]["pct_no_prazo"]

# Sazonalidade do SLA
_saz_sla = (
    fe.groupby("mes")
    .agg(pct_no_prazo=("entregue_no_prazo","mean"), n_pedidos=("id_pedido","nunique"))
    .reset_index()
)
_saz_sla["pct_no_prazo"] = pd.to_numeric(_saz_sla["pct_no_prazo"], errors="coerce") * 100
_variacao_sazonal = _saz_sla["pct_no_prazo"].max() - _saz_sla["pct_no_prazo"].min()
_pior_mes_idx     = _saz_sla["pct_no_prazo"].idxmin()
_pior_mes_sla     = _saz_sla.loc[_pior_mes_idx, "pct_no_prazo"]
_gargalo_sazonal  = (
    _saz_sla.loc[_pior_mes_idx, "n_pedidos"] >
    _saz_sla["n_pedidos"].quantile(0.6)
)

# Ponto de inflexão (reutiliza função se disponível, senão estima)
try:
    _vol_inflexao = vol_inflexao
    _delta_lt     = delta_lt
except NameError:
    _vol_inflexao = None
    _delta_lt     = None


# Degradação regional em picos
_sla_reg_v = (
    fe.groupby(["estado_cliente", "periodo"])
    .agg(pct_no_prazo=("entregue_no_prazo", "mean"), n_pedidos=("id_pedido", "nunique"))
    .reset_index()
)
_sla_reg_v["pct_no_prazo"] = pd.to_numeric(_sla_reg_v["pct_no_prazo"], errors="coerce") * 100
_estados_min = (
    _sla_reg_v.groupby("estado_cliente")["n_pedidos"].mean()
    .pipe(lambda s: s[s >= 30].index)
)
_sla_reg_v = _sla_reg_v[_sla_reg_v["estado_cliente"].isin(_estados_min)]
_vol_trim_v = fe.groupby("periodo")["id_pedido"].nunique().sort_values(ascending=False)
_top3_pico_v = _vol_trim_v.head(3).index.tolist()
_sla_hist_v = _sla_reg_v.groupby("estado_cliente")["pct_no_prazo"].mean()
_sla_pico_v = (
    _sla_reg_v[_sla_reg_v["periodo"].isin(_top3_pico_v)]
    .groupby("estado_cliente")["pct_no_prazo"].mean()
)
_deg_v = (_sla_pico_v - _sla_hist_v).dropna()
_n_estados_colapso = int((_deg_v < -10).sum())
s_regional = "⚠️  COLAPSO REGIONAL" if _n_estados_colapso >= 3 else "⚠️  RISCO PONTUAL" if _n_estados_colapso >= 1 else "✅ OK"

# ─── Semáforos ────────────────────────────────────────────────────────────────
s_frete = "⚠️  RISCO" if _r_vol_frete > 0.3  else "✅ OK"
s_lead  = "⚠️  RISCO" if _r_lt > 0.3          else "✅ OK"
s_canc  = "⚠️  RISCO" if _r_vol_canc > 0.3 and _p_canc < 0.05 else "✅ OK"
s_atr   = "⚠️  RISCO" if _r_vol_atr  > 0.3 and _p_atr  < 0.05 else "✅ OK"
s_saz   = "⚠️  SAZONAL"    if _gargalo_sazonal     else           "⚠️  ESTRUTURAL" if _variacao_sazonal > 10 else "✅ OK"
s_regional = "⚠️  COLAPSO REGIONAL" if _n_estados_colapso >= 3 else "⚠️  RISCO PONTUAL" if _n_estados_colapso >= 1 else "✅ OK"

# ─── Sinal geral ──────────────────────────────────────────────────────────────
_n_alertas = sum([
    _r_vol_frete > 0.3,
    _r_lt > 0.3,
    _r_vol_canc > 0.3 and _p_canc < 0.05,
    _r_vol_atr  > 0.3 and _p_atr  < 0.05,
    _variacao_sazonal > 10,
    _n_estados_colapso >= 1,
])
if _n_alertas == 0:
    sinal = "✅ OPERAÇÃO ESCALÁVEL — sem sinais de ruptura"
elif _n_alertas <= 2:
    sinal = "⚠️  ATENÇÃO — ESCALABILIDADE LIMITADA COM RISCOS IDENTIFICADOS"
else:
    sinal = "🔴 RISCO — OPERAÇÃO FRÁGIL SOB CRESCIMENTO"

# ─── Impressão ────────────────────────────────────────────────────────────────
print("=" * 65)
print("SÍNTESE — BLOCO 3: ESCALABILIDADE OPERACIONAL")
print("=" * 65)

print(f"""
[ CUSTO UNITÁRIO ]
  Frete médio por pedido    : R$ {_frete_medio_geral:.2f}
  Frete como % da receita   : {_pct_frete_receita:.1f}%
  Correlação vol × frete    : {_r_vol_frete:.2f} — {s_frete}

[ LEAD TIME E SLA ]
  Lead time médio           : {_lead_medio_geral:.1f} dias
  Lead time P90             : {_lead_p90:.1f} dias
  % Entregue no prazo       : {_pct_no_prazo:.1f}%
  Correlação vol × lead     : {_r_lt:.2f} — {s_lead}

[ FALHAS OPERACIONAIS ]
  Taxa de cancelamento      : {_taxa_canc:.1f}%
  Taxa de atraso            : {_taxa_atr:.1f}%
  Corr. vol × cancelamento  : {_r_vol_canc:.2f} — {s_canc}
  Corr. vol × atraso        : {_r_vol_atr:.2f} — {s_atr}

[ PONTO DE INFLEXÃO ]
  Volume limiar             : {"~{:,.0f} ped/mês".format(_vol_inflexao) if _vol_inflexao else "Não detectado"}
  Degradação acima do limiar: {"+{:.1f}d".format(_delta_lt) if _delta_lt else "N/A"}

[ NATUREZA DOS GARGALOS ]
  Pior estado (SLA)         : {_pior_estado} ({_pior_sla_est:.1f}%)
  Variação sazonal do SLA   : {_variacao_sazonal:.1f}pp — {s_saz}

[ DEGRADAÇÃO REGIONAL ]
  Estados com colapso em pico (>10pp): {_n_estados_colapso} — {s_regional}
""")

print("=" * 65)
print("VEREDICTO PARCIAL DO BLOCO 3")
print("=" * 65)
print(f"""
Sinal geral: {sinal}

Próximo passo → Bloco 4: riscos ocultos que não aparecem nas métricas agregadas.
""")


---
*Próximo notebook: `04_EDA_riscos_ocultos.ipynb` — Quais riscos um comprador herdaria ao adquirir o ativo?*

> Esta análise faz parte do **Projeto Fictus**, conduzido pela Lufi Data Consulting. Os três módulos analíticos — Vendas, Logística e Finanças — compõem a base do Relatório de Recomendação de Aquisição.
